# Grahak Gyaan

## A look at the data, the patterns, and a few model ideas

This mini-project explores 200 customer records using age, gender, annual income, and the spending score already included in the dataset. It walks through the data, compares a few simple models, and takes an exploratory look at K-means clustering.

**A note about scope:** There is no transaction history, date information, campaign response, or revenue in this file. The results describe this sample only; they cannot tell us what caused a customer's score, what someone may buy next, or whether a marketing campaign would work.

## How the analysis is organized

1. Check the file, summarize its fields, and look at the main distributions, relationships, and possible outliers.
2. Compare spending-score regression with a simple mean baseline. Use cross-validation on the training data, then check performance on a separate test set.
3. See whether customer details can approximate the score bands chosen for this project. The bands are hand-set cutoffs, not naturally discovered customer types.
4. Explore K-means groups and silhouette scores. These can help us notice patterns, but the groups need further validation before they can guide a business decision.

Customer ID is only an identifier, so it is left out of the models. Preprocessing is kept inside each scikit-learn pipeline so the test data do not influence training.

## 1. Set up the analysis

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    accuracy_score, adjusted_rand_score, balanced_accuracy_score, classification_report,
    ConfusionMatrixDisplay, f1_score, mean_absolute_error,
    mean_squared_error, r2_score, silhouette_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

SEED = 42
sns.set_theme(style="whitegrid", palette="deep")
pd.set_option("display.max_columns", 20)


## 2. Open and check the dataset

In [ ]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "Mall_Customers.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from app.data_validation import (
    SEGMENT_STRATEGIES, SPENDING_SCORE_BINS, SPENDING_SEGMENTS,
    validate_customer_data,
)

data_path = PROJECT_ROOT / "data" / "Mall_Customers.csv"
customers = validate_customer_data(pd.read_csv(data_path))
customers["Spending_Category"] = pd.cut(
    customers["Spending Score (1-100)"],
    bins=SPENDING_SCORE_BINS,
    labels=SPENDING_SEGMENTS,
    right=False,
)
print(f"Loaded and checked {len(customers)} customer records from {data_path.name}.")
display(customers.head())

## 3. Check the data quality and fields

In [ ]:
quality_checks = pd.DataFrame({
    "Check": ["Rows", "Columns", "Missing values", "Duplicate rows", "Duplicate customer IDs"],
    "Result": [
        len(customers), len(customers.columns), int(customers.isna().sum().sum()),
        int(customers.duplicated().sum()), int(customers["CustomerID"].duplicated().sum()),
    ],
})
display(quality_checks)
display(customers.dtypes.rename("Data type").to_frame())
display(customers.describe(include="all").T)

The checks flag missing required values, repeated or blank IDs, invalid numbers, and ages, incomes, or scores outside the expected range. Passing these checks tells us the file is consistent; it does not prove that the sample represents all mall customers or that every source value is correct. Customer IDs are labels, not measurements.

## 4. Take a look at each field

In [ ]:
numeric_columns = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, column in zip(axes, numeric_columns):
    sns.histplot(data=customers, x=column, bins=15, ax=axis, color="#4c8ebf")
    axis.set(title=f"{column}: distribution", ylabel="Customer count")
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
customers["Gender"].value_counts().reindex(["Female", "Male"]).plot.bar(ax=axes[0], color=["#8e79c6", "#4c8ebf"])
axes[0].set(title="Customers by gender", ylabel="Customer count", xlabel="")
customers["Spending_Category"].value_counts().reindex(SPENDING_SEGMENTS).plot.bar(ax=axes[1], color="#6e9f8a")
axes[1].set(title="Customers in each spending-score band", ylabel="Customer count", xlabel="Spending-score band")
plt.tight_layout()
plt.show()

## 5. Compare fields and customer groups

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=customers, x="Annual Income (k$)", y="Spending Score (1-100)", hue="Spending_Category", hue_order=SPENDING_SEGMENTS, ax=axes[0])
axes[0].set(title="Income and spending score")
sns.boxplot(data=customers, x="Spending_Category", y="Age", order=SPENDING_SEGMENTS, ax=axes[1])
axes[1].set(title="Age by spending-score band", xlabel="Spending-score band")
plt.tight_layout()
plt.show()

group_summary = customers.groupby("Spending_Category", observed=False).agg(
    Customers=("CustomerID", "count"),
    Average_Age=("Age", "mean"),
    Average_Income=("Annual Income (k$)", "mean"),
    Average_Score=("Spending Score (1-100)", "mean"),
)
display(group_summary.round(1))
display(customers[numeric_columns].corr().round(2))

## 6. Review possible outliers (don't remove them automatically)

In [ ]:
outlier_rows = []
for column in numeric_columns:
    q1, q3 = customers[column].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    flags = customers.loc[~customers[column].between(lower, upper), ["CustomerID", column]]
    for row in flags.itertuples(index=False):
        outlier_rows.append({"Field": column, "Customer ID": row[0], "Value": row[1], "IQR lower": lower, "IQR upper": upper})
display(pd.DataFrame(outlier_rows))

The IQR rule is a way to flag values for a closer look, not a way to prove they are errors. Keep plausible records unless the data owner confirms a problem or there is a sound reason to handle them differently. In particular, verify the two high-income values before drawing conclusions; don't remove them just to improve a model score.

## 7. Model question A: can we estimate the recorded spending score?

**Question:** Can age, gender, and annual income help estimate the spending score for a customer the model has not seen?

Set aside 20% of the records for a final check. On the remaining data, compare a mean-only baseline, linear regression, and a shallow decision tree using five-fold cross-validation. Choose using mean absolute error (MAE), then evaluate the selected model and baseline on the untouched test set. MAE is measured in score points; R² can be negative. With only 200 records, these results are an initial comparison, not a dependable forecast.

Look at the holdout results alongside the cross-validation range and the baseline rather than relying on one number. In this sample, the available fields do not estimate the spending score accurately enough for practical use. A model would need to be tested on new, relevant data before it could be considered for operational decisions.

In [ ]:
feature_columns = ["Age", "Gender", "Annual Income (k$)"]
X_reg, y_reg = customers[feature_columns], customers["Spending Score (1-100)"]
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(X_reg, y_reg, test_size=0.2, random_state=SEED)
reg_preprocessor = ColumnTransformer([
    ("gender", OneHotEncoder(handle_unknown="ignore"), ["Gender"]),
    ("numeric", StandardScaler(), ["Age", "Annual Income (k$)"]),
])
regression_models = {
    "Mean baseline": DummyRegressor(strategy="mean"),
    "Linear regression": LinearRegression(),
    "Shallow decision tree": DecisionTreeRegressor(max_depth=3, min_samples_leaf=5, random_state=SEED),
}
reg_cv = KFold(n_splits=5, shuffle=True, random_state=SEED)
reg_rows = []
for name, estimator in regression_models.items():
    model = Pipeline([("prepare", reg_preprocessor), ("model", estimator)])
    scores = cross_validate(model, X_reg_train, y_reg_train, cv=reg_cv, scoring="neg_mean_absolute_error")
    reg_rows.append({"Model": name, "CV MAE": -scores["test_score"].mean(), "CV MAE SD": scores["test_score"].std()})
regression_cv_results = pd.DataFrame(reg_rows).sort_values("CV MAE")
display(regression_cv_results.round(2))
best_regression_name = regression_cv_results.iloc[0]["Model"]
print(f"Chosen by training-data cross-validation (MAE): {best_regression_name}")
regression_test_rows = []
for name in [best_regression_name, "Mean baseline"]:
    model = Pipeline([("prepare", reg_preprocessor), ("model", regression_models[name])])
    model.fit(X_reg_train, y_reg_train)
    prediction = model.predict(X_reg_test)
    regression_test_rows.append({"Model": name, "Test MAE": mean_absolute_error(y_reg_test, prediction), "Test RMSE": mean_squared_error(y_reg_test, prediction) ** 0.5, "Test R2": r2_score(y_reg_test, prediction)})
regression_test_results = pd.DataFrame(regression_test_rows).set_index("Model")
display(regression_test_results.round(3))
selected_regression = Pipeline([("prepare", reg_preprocessor), ("model", regression_models[best_regression_name])]).fit(X_reg_train, y_reg_train)
regression_prediction = selected_regression.predict(X_reg_test)
fig, axis = plt.subplots(figsize=(5, 5))
axis.scatter(y_reg_test, regression_prediction, alpha=0.75)
axis.plot([0, 100], [0, 100], "--", color="gray")
axis.set(xlim=(0, 100), ylim=(0, 100), xlabel="Recorded score", ylabel="Estimated score", title="Estimates for held-out customers")
plt.tight_layout(); plt.show()

## 8. Model question B: can we approximate the chosen score bands?

The labels come from the spending score and the cutoffs used in the dashboard. The score itself is left out of the predictors to avoid giving the model the answer. This test asks whether age, gender, and income can approximate those cutoffs; it does **not** show that the bands are natural customer types or useful business outcomes.

Compare a most-frequent-class baseline with a shallow decision tree. Use stratified five-fold cross-validation on the training data, then check accuracy and macro F1 on the held-out records. Macro F1 and each group's recall help show where overall accuracy may hide mistakes.

In [ ]:
X_class = customers[feature_columns]
y_class = customers["Spending_Category"]
X_class_train, X_class_test, y_class_train, y_class_test = train_test_split(X_class, y_class, test_size=0.2, random_state=SEED, stratify=y_class)
class_preprocessor = ColumnTransformer([
    ("gender", OneHotEncoder(handle_unknown="ignore"), ["Gender"]),
    ("numeric", StandardScaler(), ["Age", "Annual Income (k$)"]),
])
classification_models = {
    "Most-frequent baseline": DummyClassifier(strategy="most_frequent"),
    "Shallow decision tree": DecisionTreeClassifier(max_depth=3, min_samples_leaf=5, random_state=SEED),
}
class_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
class_rows = []
for name, estimator in classification_models.items():
    model = Pipeline([("prepare", class_preprocessor), ("model", estimator)])
    scores = cross_validate(model, X_class_train, y_class_train, cv=class_cv, scoring={"accuracy": "accuracy", "macro_f1": "f1_macro"})
    class_rows.append({"Model": name, "CV accuracy": scores["test_accuracy"].mean(), "CV accuracy SD": scores["test_accuracy"].std(), "CV macro F1": scores["test_macro_f1"].mean()})
display(pd.DataFrame(class_rows).round(3))
class_model = Pipeline([("prepare", class_preprocessor), ("model", classification_models["Shallow decision tree"])]).fit(X_class_train, y_class_train)
class_prediction = class_model.predict(X_class_test)
classification_test_results = pd.DataFrame([{"Test accuracy": accuracy_score(y_class_test, class_prediction), "Test macro F1": f1_score(y_class_test, class_prediction, average="macro"), "Balanced accuracy": balanced_accuracy_score(y_class_test, class_prediction)}], index=["Shallow decision tree"])
display(classification_test_results.round(3))
print(classification_report(y_class_test, class_prediction, labels=SPENDING_SEGMENTS, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_class_test, class_prediction, labels=SPENDING_SEGMENTS, cmap="Blues")
plt.title("Held-out spending-score groups"); plt.tight_layout(); plt.show()

## 9. Explore customer groups with K-means

The score bands above use fixed cutoffs. K-means takes a different approach and groups records by their distance across selected fields. Age, annual income, and spending score are standardized so their different units do not dominate the result. Compare a small, preselected range of cluster counts using silhouette score, then review the resulting profiles.

Silhouette is a measure of how separated the groups are in this dataset. It does not show that they are stable, useful, or ready to act on. Treat these clusters as a starting point for further analysis, not a validated customer segmentation.

In [ ]:
cluster_features = ["Age", "Annual Income (k$)", "Spending Score (1-100)"]
cluster_data = StandardScaler().fit_transform(customers[cluster_features])
cluster_rows = []
for k in range(2, 7):
    labels = KMeans(n_clusters=k, random_state=SEED, n_init=20).fit_predict(cluster_data)
    cluster_rows.append({"Groups": k, "Silhouette score": silhouette_score(cluster_data, labels)})
cluster_scores = pd.DataFrame(cluster_rows)
display(cluster_scores.round(3))
selected_cluster_count = int(cluster_scores.loc[cluster_scores["Silhouette score"].idxmax(), "Groups"]) # type: ignore
cluster_model = KMeans(n_clusters=selected_cluster_count, random_state=SEED, n_init=20)
cluster_labels = cluster_model.fit_predict(cluster_data)
print(f"Exploratory K-means run with {selected_cluster_count} groups; numbers are labels, not customer-value tiers.")
stability = [adjusted_rand_score(cluster_labels, KMeans(n_clusters=selected_cluster_count, random_state=seed, n_init=20).fit_predict(cluster_data)) for seed in range(10)]
print(f"Across 10 refits, adjusted Rand index averaged {np.mean(stability):.3f} (lowest {np.min(stability):.3f}).")
clustered = customers.assign(Cluster=cluster_labels)
display(clustered.groupby("Cluster")[cluster_features].mean().round(1))
sns.scatterplot(data=clustered, x="Annual Income (k$)", y="Spending Score (1-100)", hue="Cluster", palette="tab10")
plt.title("Exploratory K-means groups"); plt.tight_layout(); plt.show()

## 10. Simple engagement ideas for the dashboard

The dashboard pairs the hand-set score bands with a few simple engagement ideas. These are examples for the presentation, not recommendations supported by campaign results. The dataset cannot tell us discount response, revenue, or expected return. To assess a real campaign, we would need information such as who saw the offer, who accepted, the cost and margin, and a fair way to compare outcomes (for example, a randomized holdout).

In [ ]:
engagement_examples = customers[["CustomerID", "Spending Score (1-100)", "Spending_Category"]].copy()
engagement_examples["Illustrative idea"] = engagement_examples["Spending_Category"].map(SEGMENT_STRATEGIES)
display(engagement_examples.head(10))

## 11. What to take away—and what not to assume

- These findings describe this dataset; they should not be generalized to every customer at the mall.
- IQR flags are prompts to review records, not proof that those records are wrong.
- Read cross-validation, holdout, and baseline results together. The sample is small, so the scores are uncertain and are not production-performance claims.
- The classifier tries to reproduce cutoffs chosen for this analysis; it does not validate those cutoffs.
- K-means profiles are exploratory. Check whether groups remain useful on other data before naming them or making decisions from them.
- There are no transactions, dates, campaign responses, or revenue here, so the analysis cannot estimate campaign lift, customer lifetime value, or financial impact.

In [ ]:
print(f"The model selected by cross-validation was {best_regression_name}.")
print(f"Its held-out MAE was {regression_test_results.loc[best_regression_name, 'Test MAE']:.2f} points; the mean baseline MAE was {regression_test_results.loc['Mean baseline', 'Test MAE']:.2f}.")
print(f"The score-band classifier reached {classification_test_results.loc['Shallow decision tree', 'Test accuracy']:.0%} test accuracy (macro F1 {classification_test_results.loc['Shallow decision tree', 'Test macro F1']:.3f}).")
print(f"K-means selected {selected_cluster_count} groups from the tested range; its silhouette score was {cluster_scores.loc[cluster_scores['Groups'] == selected_cluster_count, 'Silhouette score'].iloc[0]:.3f}.")
print("These exploratory results describe a small sample; they do not forecast customer value or campaign performance.")